# Bayesian optimisation coursework: scaling up a cell-culture process

This notebook is the only file you work in. In it you:

1. read the background, the problem and the rules,
2. write your algorithm (the `%%writefile` cell saves it to `algorithms/my_algorithm.py`),
3. save versions of your algorithm as you improve it, and
4. compare any of your versions against the two reference algorithms.

**You submit only `algorithms/my_algorithm.py`.**

## Background: why scale-up is hard

Most modern biopharmaceuticals, such as monoclonal antibodies, are made by mammalian cells (typically CHO cells) grown in
bioreactors. The cells grow, consume nutrients (mainly **glucose** and **glutamine**) and secrete the product. The amount of
product per volume at the end of the run, the **titer**, largely decides how much a process costs.

Process development has to choose the cell line, temperature, pH, the starting medium, the feeding strategy and when to shift
temperature or pH during the run. These choices
interact in non-linear ways:

- **Too little** glucose or glutamine and the cells starve. **Too much** and they make **lactate** and **ammonia**, by-products
  that inhibit growth and productivity.
- Each **cell line** has its own growth rate, productivity, metabolism and preferred pH. The best recipe for one clone isn't
  the best for another.
- **Temperature** and **pH** speed up or slow down growth, death and production, each with its own optimum.

The catch is **scale**. Recipes are developed in small vessels because they are cheap and fast, but the product is made in large
ones. A large bioreactor isn't simply a bigger small one: mixing times, oxygen transfer, CO₂ removal, shear stress and
gradients of pH, temperature and nutrients all change with scale. **The recipe that is optimal at small scale is often not
optimal at large scale**, and experiments at large scale are expensive and slow. This is the scale-up problem, and a major
cost and risk in bioprocess development.

## Your task: choose the scale, then the experiments

You have **30 candidate clones** (`celltype_1` … `celltype_30`) and three scales in a simulated lab. Each scale runs a fixed
batch of experiments in parallel, with its own rules:

| Scale | What it is | Batch per call | What you can set | Cost per experiment |
| --- | --- | --- | --- | --- |
| `MTP` | 24-well microtiter plate (~1 mL), high-throughput screening | **24 wells** | **only the clone per well.** Temperature, pH and the starting medium are one setting for the whole plate, and there are **no feeds** and **no temperature/pH shifts** | $10 |
| `MBR` | mini bioreactors (~100 mL) | **4 reactors** | everything, per reactor | $500 |
| `Pilot` | pilot-scale bioreactors (~3 L) | **1 reactor** | everything, per reactor | $3,000 |

Each experiment is a 14-day (336 h) **fed-batch** culture: you set the starting temperature, pH and medium, and two feeds, at day 5
(120 h) and day 10 (240 h). At each feed you can also **change the temperature and pH** for the rest of the run (a common trick:
grow the cells first, then shift to conditions that favour production). Plates can't be fed or shifted. The lab returns the
final product titer, with measurement noise.

**Goal: find the clone and recipe with the highest titer at `Pilot` scale, and find it early.** Only Pilot titers count for your
score, and they are checked at 33 %, 66 % and 100 % of the budget (see the rules). So in every step your algorithm decides two
things:

1. **Which scale** to run next. Plates screen many clones very cheaply, but only under one fixed condition with no feeding, and
   they use up 24 experiments at once. Mini bioreactors test full recipes at medium cost. Pilot runs are what you're graded on,
   but they're expensive and run one at a time.
2. **Which experiments** to put in that batch, with exactly the right number of them.

Small scales are biased: they tell you something about Pilot, but not everything. A good algorithm uses the cheap scales to learn
where to look and spends the expensive Pilot runs wisely. This is **multi-fidelity Bayesian optimisation**.

How the 30 clones are generated (random parameters, each with its own optimal temperature and pH) is described in the student
[README.md](README.md#how-the-clones-are-made). Use it to understand the problem, but **don't fit your algorithm to it**: the
grading lab has new clones.

## The rules

| Rule | Value |
| --- | --- |
| Budget per run | **$25,000** (`lab.budget`, printed below). Each experiment (well or reactor) costs its scale's price |
| Max experiments per run | **300** (`lab.max_experiments`) wells + reactors in total, including the initial design. A plate uses 24 of them |
| Batches | every call `lab(x)` runs **one scale** with **exactly** `lab.batch_sizes[scale]` rows: 24 / 4 / 1. A batch runs completely or not at all |
| Plates | all 24 wells share temperature, pH and starting medium; feeds must be 0 and there are no temperature/pH shifts. Build plates with `lab.plate(...)` |
| Time limit | **5 minutes per run** (one seed). At 5 minutes your algorithm is **stopped** and graded on the Pilot titers found up to then, so any budget left is wasted |
| Score | the best titer measured in the **`Pilot`** reactor once **33 %**, **66 %** and **100 %** of the budget is spent (`lab.score_checkpoints`, in $), **summed**. A checkpoint with no Pilot result yet counts **0**, so run a Pilot experiment early. You get a warning when a run misses a checkpoint |
| Seeds | your algorithm runs once per seed and the results are averaged. Seed **all** your randomness from the `seed` argument |
| Allowed imports | `numpy`, `scipy`, `random`, `sobol_seq`, `time`, `datetime`, `typing`. Nothing else, not even `sklearn` |
| Lab interface | use only the public attributes and methods of `lab` listed below; the lab is read-only. Anything starting with `_` (e.g. `lab._simulate`) is rejected at submission, and reading files or the lab configuration, or hard-coding clone names, is flagged for the GTA |
| Names | set `STUDENT_NAMES` to the full names of everyone in your group |

### ⚠️ Grading uses a different lab

Your algorithm is graded on a **hidden lab** that differs from this practice one. The GTA can for example change:

- **the clones**: 30 different cell lines with different growth, metabolism, productivity and pH/temperature optima, so the best
  clone and recipe will be somewhere else;
- **the seeds**: different measurement noise, and a different initial design;
- **the initial design**: different starting experiments, and possibly a different number of plates and reactor batches;
- **the lab settings**: budget, maximum number of experiments, reactor costs and batch sizes;
- and more.

So **don't tune to this practice lab**: no hard-coded recipes, cell types or numbers. Read every setting from the `lab` object
(`lab.budget`, `lab.max_experiments`, `lab.batch_sizes`, `lab.reactor_costs`, `lab.celltypes`, `lab.bounds`, ...) and use `initial_x`/`initial_y`
whatever their length. Test your algorithm on several seeds, not just one.

In [1]:
# Reload the course code automatically if it changes (e.g. after an update), without restarting the kernel.
%load_ext autoreload
%autoreload 2

from digital_lab import SEEDS, DigitalLab
from digital_lab import check_my_algorithm, list_versions, run_comparison, save_version, show_results, show_scale_choices

# The settings, read from a lab object the same way your algorithm reads them.
lab = DigitalLab(seed=SEEDS[0])
print(f"Clones:                  {len(lab.celltypes)}")
print(f"Budget per run:          ${lab.budget:,}")
print(f"Cost per experiment:     { {r: f'${c:,}' for r, c in lab.reactor_costs.items()} }")
print(f"Batch per lab(x) call:   {lab.batch_sizes}  ->  cost per batch "
      f"{ {r: f'${lab.batch_cost(r):,}' for r in lab.reactor_types} }")
print(f"Max experiments per run: {lab.max_experiments} (wells + reactors, including the initial design)")
print(f"Initial design:          {lab.initial_design} batches, already run and paid for")
print(f"Score checkpoints:       {[f'${c:,.0f}' for c in lab.score_checkpoints]} spent (best Pilot titer at each, summed)")
print(f"Time limit per run:      {lab.time_limit} s")
print(f"Practice seeds:          {SEEDS}")

Clones:                  30
Budget per run:          $25,000
Cost per experiment:     {'MTP': '$10', 'MBR': '$500', 'Pilot': '$3,000'}
Batch per lab(x) call:   {'MTP': 24, 'MBR': 4, 'Pilot': 1}  ->  cost per batch {'MTP': '$240', 'MBR': '$2,000', 'Pilot': '$3,000'}
Max experiments per run: 300 (wells + reactors, including the initial design)
Initial design:          {'MTP': 1, 'MBR': 1, 'Pilot': 0} batches, already run and paid for
Score checkpoints:       ['$8,250', '$16,500', '$25,000'] spent (best Pilot titer at each, summed)
Time limit per run:      300 s
Practice seeds:          [42, 43, 44, 45, 46]


## Talking to the lab

Each experiment is one row of a NumPy **object** array (`dtype=object`) with 14 columns:

| # | Column | Values |
| --- | --- | --- |
| 0 | reactor | `"MTP"`, `"MBR"` or `"Pilot"` (`lab.reactor_types`); the same in every row of a batch |
| 1 | clone | `"celltype_1"` … `"celltype_30"` (`lab.celltypes`) |
| 2 | temperature | °C (plate: same in every well) |
| 3 | pH | (plate: same in every well) |
| 4, 5 | initial glucose, glutamine | mM (plate: same in every well) |
| 6, 7 | feed 1 (day 5) glucose, glutamine | mM (plate: must be 0) |
| 8, 9 | feed 2 (day 10) glucose, glutamine | mM (plate: must be 0) |
| 10, 11 | temperature, pH from feed 1 (day 5) on | °C, pH (plate: same as columns 2, 3) |
| 12, 13 | temperature, pH from feed 2 (day 10) on | °C, pH (plate: same as columns 2, 3) |

The `(low, high)` bounds of columns 2–13 are in `lab.bounds`, in this order, so a row has `2 + len(lab.bounds)` columns. To keep
the temperature and pH constant, repeat columns 2 and 3 in columns 10–13.

Your function is `my_algorithm(initial_x, initial_y, lab, seed)`:

- `initial_x`, `initial_y`: the initial design, already run and paid for. By default one plate (24 random clones under one random
  condition) and one batch of 4 mini bioreactors.
- `lab(x)`: runs one batch at one scale and returns `titers, spent`. `x` must have exactly `lab.batch_sizes[reactor]` rows, all with
  the same reactor. If the budget or the experiment allowance can't cover the whole batch, nothing runs and `titers` is empty.
- `lab.plate(clones, temperature, ph, glucose, glutamine)` builds a valid plate from a list of 24 clones (repeats allowed).
- `lab.affordable_reactors()` lists the scales that can still run a batch (empty when the run is over), and `lab.can_run(reactor)`
  checks one. `lab.batch_cost(reactor)` is the cost of a full batch.
- Other attributes: `lab.budget`, `lab.spent`, `lab.remaining_budget`, `lab.max_experiments`, `lab.n_experiments`,
  `lab.remaining_experiments`, `lab.batch_sizes`, `lab.reactor_costs`, `lab.bounds`, `lab.celltypes`, `lab.objective_reactor`,
  `lab.screening_reactor`, `lab.feed_times` (h), `lab.score_checkpoints` (money spent, in $, at which you're scored),
  `lab.time_limit`, `lab.time_remaining`.
- `seed`: use it, e.g. `np.random.default_rng(seed)`.

Keep both limits in mind: a plate is cheap in money but uses 24 experiments, a Pilot run is expensive but uses only 1. If you
run out of experiments with money left over, that money is wasted. And keep the checkpoints in mind: check
`lab.spent + lab.batch_cost(reactor)` against `lab.score_checkpoints` so you have a Pilot result before each one.

The two reference algorithms are working examples: [algorithms/reference_screen_then_confirm.py](algorithms/reference_screen_then_confirm.py) is a screening
funnel without a model (an extra plate, then mini bioreactors, then Pilot): see how it builds a valid batch for every scale and
keeps budget and experiments for Pilot, and [algorithms/reference_mfbo.py](algorithms/reference_mfbo.py) is a simplified multi-fidelity
Gaussian-process + Expected Improvement approach that also picks the scale. It uses `sklearn`, which **you may not**, so you'll have
to write your own GP (with `numpy`/`scipy`).

## Your algorithm

Edit the cell below and run it. It saves the cell to `algorithms/my_algorithm.py`, overwriting it. The starting point is random:
a random affordable scale each time, with a valid random batch for it.

In [2]:
%%writefile algorithms/my_algorithm.py
"""My algorithm. Written from project.ipynb — edit it there, this file is overwritten."""

import numpy as np

# Full names (like onImperial ID card) and CIDs of everyone in your group. Your results are reported under these names.
STUDENT_NAMES = ["TODO Firstname Lastname CID", "TODO Firstname Lastname CID"]


def my_algorithm(initial_x, initial_y, lab, seed):
    """Spend the lab's budget to find the highest Pilot-scale titer.

    Parameters
    ----------
    initial_x : numpy.ndarray
        Initial design rows (already run and paid for).
    initial_y : numpy.ndarray
        Titers observed for `initial_x`.
    lab : DigitalLab
        Call `titers, spent = lab(x)` with one full batch at one scale
        (`lab.batch_sizes[reactor]` rows).
    seed : int
        Seed all your randomness from this.
    """
    rng = np.random.default_rng(seed)
    while lab.affordable_reactors():  # empty once no scale can run another batch
        reactor = rng.choice(lab.affordable_reactors())
        size = lab.batch_sizes[reactor]
        if reactor == lab.screening_reactor:
            # Plate: one clone per well, one shared temperature, pH and medium, no feeds.
            clones = rng.choice(lab.celltypes, size=size, replace=size > len(lab.celltypes))
            temperature, ph, glucose, glutamine = [rng.uniform(low, high) for low, high in lab.bounds[:4]]
            x = lab.plate(clones, temperature, ph, glucose, glutamine)
        else:
            # Bioreactors: every run has its own clone, conditions, feeds and temperature/pH shifts.
            x = np.empty((size, 2 + len(lab.bounds)), dtype=object)
            x[:, 0] = reactor
            x[:, 1] = rng.choice(lab.celltypes, size=size)
            for i, (low, high) in enumerate(lab.bounds):
                x[:, 2 + i] = rng.uniform(low, high, size=size).tolist()
        titers, spent = lab(x)

Overwriting algorithms/my_algorithm.py


In [3]:
# Checks the submission rules (imports, function name, STUDENT_NAMES, no private lab._ attributes).
# Fix any warning before you submit.
check_my_algorithm("algorithms/my_algorithm.py")

  - Set STUDENT_NAMES to a list with your full names, e.g. ['Sam Stricker 006547054', 'Antonio del Rio Chanona 001234567'].


False

## Save a version

When you have a version worth keeping, save it under a name (letters, digits, `_` and `-`). It's copied to
`algorithms/saved/<name>.py`, and you can then compare against it below while you keep changing `my_algorithm`. Saving under an
existing name overwrites that version.

In [4]:
# save_version("v1_random_search")   # uncomment, pick a name, and run to save the current my_algorithm
print("Saved versions:", list_versions())

Saved versions: []


## Compare

Choose what to run in `ALGORITHMS`: `"my_algorithm"`, `"reference_screen_then_confirm"`, `"reference_mfbo"`, and/or any saved version names
(see `list_versions()` above).

**Results are saved** in `results/` and reused: an algorithm only runs again if its code, the seed or the lab changed. So after
you edit `my_algorithm`, only it re-runs, while the references and your saved versions load instantly (the output says
`ran` or `saved result` for each seed). Pass `rerun=True` to run everything again anyway.

The first run of everything on all seeds takes a while (mostly `reference_mfbo`), so while you develop you can also use fewer
seeds (e.g. `SEEDS[:2]`). You'll get a **warning** if a run is stopped at the time limit, or if it had no Pilot result yet at a scoring checkpoint.

In [5]:
ALGORITHMS = ["my_algorithm", "reference_screen_then_confirm", "reference_mfbo"]
RUN_SEEDS = SEEDS

runs = run_comparison(ALGORITHMS, seeds=RUN_SEEDS)  # add rerun=True to ignore saved results

my_algorithm on 5 seed(s):
  seed 42: score 1929.9 (best Pilot titer at 33/66/100 % of budget: 139.0 / 895.5 / 895.5), 0.3 s (ran)
  seed 43: score 15262.9 (best Pilot titer at 33/66/100 % of budget: 5087.6 / 5087.6 / 5087.6), 0.4 s (ran)
  seed 44: score 801.6 (best Pilot titer at 33/66/100 % of budget: 171.8 / 171.8 / 458.1), 0.3 s (ran)
  seed 45: score 3842.1 (best Pilot titer at 33/66/100 % of budget: 1280.7 / 1280.7 / 1280.7), 0.2 s (ran)
  seed 46: score 46572.2 (best Pilot titer at 33/66/100 % of budget: 15524.1 / 15524.1 / 15524.1), 0.3 s (ran)
reference_screen_then_confirm on 5 seed(s):
  seed 42: score 10444.1 (best Pilot titer at 33/66/100 % of budget: 2404.9 / 3903.9 / 4135.4), 0.0 s (ran)
  seed 43: score 29520.7 (best Pilot titer at 33/66/100 % of budget: 9240.3 / 9240.3 / 11040.1), 0.1 s (ran)
  seed 44: score 30547.6 (best Pilot titer at 33/66/100 % of budget: 9535.0 / 10506.3 / 10506.3), 0.0 s (ran)
  seed 45: score 23398.3 (best Pilot titer at 33/66/100 % of budget: 

In [6]:
show_results(runs)

┌───────────────────────────────┬───────────────────┬─────────────────┬─────────────────┬─────────────────┬──────────────────┐
│ algorithm                     ┆ score             ┆ best at 33 %    ┆ best at 66 %    ┆ best at 100 %   ┆ max run time [s] │
╞═══════════════════════════════╪═══════════════════╪═════════════════╪═════════════════╪═════════════════╪══════════════════╡
│ my_algorithm                  ┆ 13681.7 ± 17234.3 ┆ 4440.6 ± 5830.5 ┆ 4591.9 ± 5725.8 ┆ 4649.2 ± 5682.6 ┆ 0.4              │
│ reference_screen_then_confirm ┆ 24722.9 ± 7581.5  ┆ 7317.8 ± 2611.2 ┆ 8254.1 ± 2540.6 ┆ 9151.1 ± 2563.4 ┆ 0.1              │
│ reference_mfbo                ┆ 18862.4 ± 5984.8  ┆ 2614.3 ± 2908.5 ┆ 7099.8 ± 3935.2 ┆ 9148.3 ± 2086.6 ┆ 24.4             │
└───────────────────────────────┴───────────────────┴─────────────────┴─────────────────┴─────────────────┴──────────────────┘


**Reading the results**

- **score**: the best Pilot titer at 33 % + at 66 % + at 100 % of the budget, mean ± std over seeds. Higher is better. This is
  your score: the ranking uses only this. The columns **best at 33 / 66 / 100 %** show the three parts (a missed checkpoint is 0).
- **The plots** show, for each scale side by side (MTP, MBR, Pilot), the best titer you actually *measured* there so far, mean ±
  std over seeds: once against money spent and once against number of experiments. Before a seed's first run at a scale it
  counts as 0, so you can also see *when* each algorithm started using that scale. The dotted lines on the Pilot panel are the
  scoring checkpoints.

### Which scale did each algorithm choose?

One row per seed, one square per batch (lab call), colored by scale. Batch 0, the initial design, isn't shown. The **red square**
marks the end of the run, labelled with why it ended: `budget`, `experiment limit`, `budget + experiment limit`, `time limit`,
`error`, or `stopped early` (your algorithm returned while it could still have run another batch; hover to see which scale). Use it to check
your scale strategy: does it screen early, use the mini bioreactors, and save enough experiments and budget for Pilot runs? To
look at only some algorithms, pass their names, e.g. `show_scale_choices(runs, ["my_algorithm"])`.

In [7]:
show_scale_choices(runs)

## Submitting

Run the `%%writefile` cell once more so the file is up to date, check that `check_my_algorithm` reports no problems, and hand in
**only** `algorithms/my_algorithm.py`.